# Score every face detector on the webcam test set (run once)

The paper's face-detection tables use the WIDER FACE + FDDB 70/20/10 test split (`train_eval_face_70_20_10.ipynb`,
`finetune_blazeface_70_20_10.ipynb`). That split is mostly crowd photos: 40% of its faces are under 10% of the image.
Knowing Eye sees one examinee at a laptop webcam. This notebook scores every face detector on the **test people** of
Knowing Eye's own webcam data, recorded for exactly that situation:

| Step | Tool |
|---|---|
| record frames per person, on a timer, under 14 conditions (looking down, head turned, dim light, a second person, an empty seat, ...) | `scripts/capture_webcam_faces.py` |
| draw the ground-truth boxes by hand, without ever seeing a detector's output | `scripts/label_webcam_faces.py` |
| put every person into train / val / test (as a whole, at random, fixed), drop near-duplicate frames, write the server view, zip | `scripts/label_webcam_faces.py --pack` |
| fine-tune BlazeFace on the train people, pick the epoch on the val people | `finetune_blazeface_webcam.ipynb` |
| score every model on the test people **once** | this notebook |

**Models** - the exact weights the paper's tables come from:
* YOLOv8n, YOLO11n, RT-DETR-l - `best.pt` from `MyDrive/knowing-eye-70-20-10/<model>/`;
* BlazeFace (fine-tuned) - `blaze_face_short_range_finetuned.tflite` from `MyDrive/knowing-eye-blazeface-70-20-10/`;
* BlazeFace (webcam-tuned) - `blaze_face_short_range_webcam.tflite` from `MyDrive/knowing-eye-blazeface-webcam/`;
* BlazeFace (original) - Google's model, as the baseline.

**Two views of every test frame**, each with the threshold the app uses there:

| View | Frames | Fixed confidence | Where in Knowing Eye |
|---|---|---|---|
| camera | as the webcam delivers them (640x480) | 0.5 (MediaPipe default, `face-tracker.ts`) | browser overlay |
| server | scaled to 480 px wide, JPEG quality 60 (`use-monitoring.ts`) | 0.62 (`face_detector.py`) | server-side proctoring |

On the server, the detector's faces are then passed to MediaPipe's face-landmark model; this notebook measures the
detector itself.

**Same protocol as the paper's tables.** YOLO / RT-DETR run at 640 px with Ultralytics' validation settings. BlazeFace
runs through MediaPipe's exact pre/post-processing (checked against MediaPipe below). Every model is scored by the same
Ultralytics-identical metric code, copied from `finetune_blazeface_70_20_10.ipynb`. On top of the paper's columns it
reports each model at the fixed confidence of each view, recall and false alarms per condition, and 95% confidence
intervals.

**Scored once.** Each model's detections are saved to Drive under the dataset's fingerprint. Running the notebook again
re-uses them instead of running the models again, so the tables and figures can be remade but the test set is only
used once. A model added later is scored once on its own. Every scoring run is appended to `scoring_log.csv`, which
belongs in the paper's appendix.

**Before running**
1. Run `python scripts/label_webcam_faces.py --pack` and upload `datasets/webcam_test.zip` to the root of Google Drive
   (`MyDrive/webcam_test.zip`).
2. The outputs of the 70/20/10 notebooks and of `finetune_blazeface_webcam.ipynb` must already be on Drive. A model
   whose weights are missing is skipped - and scored once on its own later, when they appear.
3. Runtime -> Change runtime type -> **T4 GPU** (a CPU runtime works too; RT-DETR is just slower).
4. Runtime -> Run all.

**Rule:** finish labelling and training before running this, and do not change labels, exclusions or settings after
seeing results. The test set is only honest if it is used once.

In [ ]:
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Same versions as the 70/20/10 notebooks. MediaPipe's Python API needs the EGL/GLES libraries.
    subprocess.run('apt-get -qq install -y libegl1 libgles2 > /dev/null', shell=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe==1.1.1', 'ultralytics==8.4.155'],
                   check=True)

import hashlib, json, math, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
import ultralytics
from mediapipe.tasks.python import BaseOptions, vision
from ultralytics import RTDETR, YOLO

try:   # tf.lite.Interpreter is being moved to the ai_edge_litert package
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
trapezoid = getattr(np, 'trapezoid', None) or np.trapz   # NumPy 2 renamed trapz

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('MediaPipe', mp.__version__, '| Ultralytics', ultralytics.__version__,
      '| device:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/webcam_test.zip'
DATA_ROOT = '/content/data'
YOLO_DIR = '/content/drive/MyDrive/knowing-eye-70-20-10'             # outputs of train_eval_face_70_20_10.ipynb
BLAZE_DIR = '/content/drive/MyDrive/knowing-eye-blazeface-70-20-10'   # outputs of finetune_blazeface_70_20_10.ipynb
WEBCAM_BLAZE_DIR = '/content/drive/MyDrive/knowing-eye-blazeface-webcam'   # outputs of finetune_blazeface_webcam.ipynb
OUT_DIR = '/content/drive/MyDrive/knowing-eye-webcam-test'            # everything that must survive the session

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = f'{DATA_ROOT}/webcam_test'
if not Path(DATA_DIR, 'manifest.csv').exists():
    zipfile.ZipFile(DATASET_ZIP).extractall(DATA_ROOT)   # local disk: much faster than reading from Drive
FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_yolo_labels(path):
    """YOLO 'cls cx cy w h' (normalised) -> (n, 4) xyxy normalised to the image."""
    rows = [l.split() for l in Path(path).read_text().splitlines() if l.strip()] if Path(path).exists() else []
    a = np.array([[float(v) for v in r[1:5]] for r in rows], np.float32).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2, a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], 1)


# The fingerprint ties every result to this exact set of frames and labels.
FINGERPRINT = sha256(f'{DATA_DIR}/manifest.csv')[:12]
manifest = pd.read_csv(f'{DATA_DIR}/manifest.csv', dtype=str).astype({'faces': int, 'excluded': int})
assert set(manifest['split']) == {'test'}, 'webcam_test.zip must hold test people only'
frames = manifest[manifest['excluded'] == 0].reset_index(drop=True)
for row in frames.itertuples():   # every frame and label must be exactly what was labelled and packed
    stem = Path(row.file).stem
    assert sha256(f'{DATA_DIR}/images/{row.file}') == row.image_sha256, f'{row.file} differs from the manifest'
    assert sha256(f'{DATA_DIR}/server/images/{stem}.jpg') == row.server_sha256, f'{row.file}: server view changed'
    for labels in ('labels', 'server/labels'):
        assert sha256(f'{DATA_DIR}/{labels}/{stem}.txt') == row.label_sha256, f'{row.file}: label changed'

# camera: as the webcam delivers it; server: scaled to 480 px wide, JPEG quality 60 - see the table above.
VIEWS = {'camera': dict(dir=DATA_DIR, conf=0.5), 'server': dict(dir=f'{DATA_DIR}/server', conf=0.62)}
IMAGE_PATHS = {'camera': [f'{DATA_DIR}/images/{f}' for f in frames['file']],
               'server': [f'{DATA_DIR}/server/images/{Path(f).stem}.jpg' for f in frames['file']]}
GT = [read_yolo_labels(f'{DATA_DIR}/labels/{Path(f).stem}.txt') for f in frames['file']]   # normalised: both views
PARTICIPANT = frames['participant'].to_numpy()
CONDITION = frames['condition'].to_numpy()
N_FRAMES = len(frames)
assert N_FRAMES and [len(g) for g in GT] == frames['faces'].tolist()

sizes = np.concatenate([np.sqrt((g[:, 2] - g[:, 0]) * w * (g[:, 3] - g[:, 1]) * h) / max(w, h)
                        for g, w, h in zip(GT, frames['width'].astype(int), frames['height'].astype(int))])
dropped = manifest.loc[manifest.excluded == 1, 'exclude_reason'].fillna('')
print(f'Dataset {FINGERPRINT}: {N_FRAMES} frames, {len(sizes)} faces, {frames.participant.nunique()} test people; '
      f'{dropped.str.startswith("near-duplicate").sum()} near-duplicate frame(s) dropped, '
      f'{dropped.str.startswith("labeller").sum()} excluded by the labeller. Median face size {np.median(sizes):.0%} '
      f'of the image side (the WIDER/FDDB test split: 40% of faces under 10%).')
summary = frames.groupby('condition').agg(frames=('file', 'size'), faces=('faces', 'sum'),
                                          frames_without_face=('faces', lambda f: int((f == 0).sum())))
summary

In [ ]:
# ---- Protocol: copied from the 70/20/10 notebooks and fixed before any model sees this set ----
IMGSZ, YOLO_CONF, YOLO_IOU, MAX_DET = 640, 0.001, 0.7, 300   # Ultralytics validation settings
# Fixed operating point of each view (VIEWS, above): camera 0.5 = the browser overlay's minDetectionConfidence,
# server 0.62 = the server FaceLandmarker's min_face_detection_confidence. The same for every model.
MEAN_CONF_MIN = 0.25  # 'Mean Confidence' column: mean score of detections >= 0.25, as in the paper's tables
BOOTSTRAP, SEED = 1000, 42
MIN_PARTICIPANTS_FOR_CI = 5   # fewer -> frame-level intervals (frames of one person are not independent)

GOOGLE_BLAZEFACE = ('https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/'
                    'latest/blaze_face_short_range.tflite',
                    'b4578f35940bf5a1a655214a1cce5cab13eba73c1297cd78e1a04c2380b0152f')   # = the repo's copy
pretrained = Path(BLAZE_DIR, 'pretrained', 'blaze_face_short_range.tflite')
if not pretrained.exists():
    pretrained = Path(OUT_DIR, 'pretrained', 'blaze_face_short_range.tflite')
    pretrained.parent.mkdir(parents=True, exist_ok=True)
    if not pretrained.exists():
        urllib.request.urlretrieve(GOOGLE_BLAZEFACE[0], pretrained)
assert sha256(pretrained) == GOOGLE_BLAZEFACE[1], 'original BlazeFace differs from the repository copy'

MODELS = {   # name -> (family, weights)
    'yolov8n': ('yolo', f'{YOLO_DIR}/yolov8n/best.pt'),
    'yolo11n': ('yolo', f'{YOLO_DIR}/yolo11n/best.pt'),
    'rtdetr-l': ('rtdetr', f'{YOLO_DIR}/rtdetr-l/best.pt'),
    'blazeface-webcam': ('blazeface', f'{WEBCAM_BLAZE_DIR}/blaze_face_short_range_webcam.tflite'),
    'blazeface': ('blazeface', f'{BLAZE_DIR}/blaze_face_short_range_finetuned.tflite'),
    'blazeface-pretrained': ('blazeface', str(pretrained)),
}
DISPLAY = {'yolov8n': 'YOLOv8n', 'yolo11n': 'YOLOv11n', 'rtdetr-l': 'RT-DETR-l',
           'blazeface-webcam': 'BlazeFace (webcam-tuned)', 'blazeface': 'BlazeFace (fine-tuned)',
           'blazeface-pretrained': 'BlazeFace (original)'}
TYPE = {'yolov8n': '1-stage CNN', 'yolo11n': '1-stage CNN', 'rtdetr-l': 'Transformer',
        'blazeface-webcam': '1-stage CNN', 'blazeface': '1-stage CNN', 'blazeface-pretrained': '1-stage CNN'}

AVAILABLE = {n: m for n, m in MODELS.items() if Path(m[1]).exists()}
for name in MODELS.keys() - AVAILABLE.keys():
    print(f'{DISPLAY[name]}: weights not found at {MODELS[name][1]} - skipped')
WEIGHTS_SHA = {name: sha256(weights) for name, (_, weights) in AVAILABLE.items()}
print('Scoring:', ', '.join(DISPLAY[n] for n in AVAILABLE))

## 1. Detectors and metrics

Copied from `finetune_blazeface_70_20_10.ipynb`, so every number means what it means in the paper's tables:

* **BlazeFace** - letterbox to 128x128 with one bilinear `warpAffine`, MediaPipe's 896 SSD anchors and decoding,
  weighted NMS at IoU 0.3, scores kept down to 0.01 for the PR curve.
* **Metrics** - Ultralytics 8.4's `match_predictions` / `ap_per_class` (101-point COCO AP). Precision, Recall, F1 and
  Accuracy = TP / (TP + FP + FN) are read at the confidence that maximises F1, as in the paper's Table 2.
* **At the view's fixed confidence** - only detections at or above it are kept and matched to faces at IoU >= 0.5:
  the TP / FP / FN the deployed detector produces. A frame with no face can only produce false alarms.

In [ ]:
INPUT = 128
SSD = dict(num_layers=4, min_scale=0.1484375, max_scale=0.75, strides=[8, 16, 16, 16])


def ssd_anchors():
    """(896, 3) [x_centre, y_centre, nominal_scale] in MediaPipe's anchor order."""
    n, lo, hi, strides = SSD['num_layers'], SSD['min_scale'], SSD['max_scale'], SSD['strides']
    scale = lambda i: lo + (hi - lo) * i / (n - 1)
    anchors, layer = [], 0
    while layer < n:
        scales, last = [], layer
        while last < n and strides[last] == strides[layer]:
            nxt = 1.0 if last == n - 1 else scale(last + 1)
            scales += [scale(last), math.sqrt(scale(last) * nxt)]
            last += 1
        fm = math.ceil(INPUT / strides[layer])
        anchors += [((x + 0.5) / fm, (y + 0.5) / fm, s) for y in range(fm) for x in range(fm) for s in scales]
        layer = last
    return np.array(anchors, np.float32)


ANCHORS = ssd_anchors()
assert len(ANCHORS) == 896


def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f'cannot read {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def letterbox(img_rgb, size=INPUT):
    """MediaPipe ImageToTensor(keep_aspect_ratio, BORDER_ZERO): centred square ROI, one bilinear warpAffine."""
    h, w = img_rgb.shape[:2]
    s = max(h, w) / size
    m = np.array([[1 / s, 0, (max(h, w) - w) / 2 / s], [0, 1 / s, (max(h, w) - h) / 2 / s]], np.float32)
    out = cv2.warpAffine(img_rgb, m, (size, size), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out, (w / s / size, h / s / size, m[0, 2] / size, m[1, 2] / size)


def normalize(batch_uint8):
    return batch_uint8.astype(np.float32) / 127.5 - 1.0


def decode(regressors, classificators):
    """Raw (896,16), (896,1) -> boxes xyxy (input-normalised), keypoints (896,6,2), scores."""
    r = regressors / INPUT
    cx, cy = ANCHORS[:, 0] + r[:, 0], ANCHORS[:, 1] + r[:, 1]
    boxes = np.stack([cx - r[:, 2] / 2, cy - r[:, 3] / 2, cx + r[:, 2] / 2, cy + r[:, 3] / 2], 1)
    kps = r[:, 4:].reshape(-1, 6, 2) + ANCHORS[:, None, :2]
    scores = 1 / (1 + np.exp(-np.clip(classificators[:, 0].astype(np.float64), -100, 100)))
    return boxes, kps, scores


def iou_matrix(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = lambda x: np.clip(x[:, 2] - x[:, 0], 0, None) * np.clip(x[:, 3] - x[:, 1], 0, None)
    union = area(a)[:, None] + area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def weighted_nms(boxes, kps, scores, min_score, iou_thresh=0.3):
    """MediaPipe NonMaxSuppressionCalculator, algorithm WEIGHTED, overlap IoU."""
    order = np.argsort(-scores)
    order = order[scores[order] >= min_score]
    out_b, out_k, out_s = [], [], []
    while order.size:
        sim = iou_matrix(boxes[order[:1]], boxes[order])[0]
        group = order[sim > iou_thresh]
        group = group if group.size else order[:1]
        w = scores[group][:, None]
        out_b.append((boxes[group] * w).sum(0) / w.sum())
        out_k.append((kps[group] * w[:, :, None]).sum(0) / w.sum())
        out_s.append(scores[order[0]])
        if not (sim > iou_thresh).any():
            break   # MediaPipe stops when an iteration removes nothing
        order = order[sim <= iou_thresh]
    if not out_s:
        return np.zeros((0, 4), np.float32), np.zeros((0, 6, 2), np.float32), np.zeros(0, np.float32)
    return np.array(out_b, np.float32), np.array(out_k, np.float32), np.array(out_s, np.float32)


def unletterbox(boxes, lb):
    sx, sy, px, py = lb
    return (boxes - [px, py, px, py]) / [sx, sy, sx, sy]


def detect(reg, cls, lbs, min_score=0.01, iou_thresh=0.3, top_k=300):
    """Per image: decode + weighted NMS, back to image-normalised xyxy, clipped to the image."""
    out = []
    for r, c, lb in zip(reg, cls, lbs):
        boxes, kps, scores = decode(r, c)
        # MediaPipe drops boxes with non-positive width/height before NMS.
        scores = np.where((boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1]), scores, -1.0)
        keep = np.argsort(-scores)[:top_k]
        keep = keep[scores[keep] >= min_score]
        b, _, s = weighted_nms(boxes[keep], kps[keep], scores[keep], min_score, iou_thresh)
        out.append((np.clip(unletterbox(b, lb), 0, 1), s))
    return out


def blazeface_outputs(model_path, paths):
    """Raw regressors / classificators of a BlazeFace .tflite for each image, plus its letterbox."""
    it = Interpreter(model_path=str(model_path))
    it.allocate_tensors()
    inp = it.get_input_details()[0]['index']
    out = {tuple(int(v) for v in d['shape'][1:]): d['index'] for d in it.get_output_details()}
    regs, clss, lbs = [], [], []
    for p in paths:
        x, lb = letterbox(read_rgb(p))
        it.set_tensor(inp, normalize(x[None]))
        it.invoke()
        regs.append(it.get_tensor(out[(896, 16)])[0])
        clss.append(it.get_tensor(out[(896, 1)])[0])
        lbs.append(lb)
    return np.stack(regs), np.stack(clss), lbs


IOU_T = np.linspace(0.5, 0.95, 10)


def match_predictions(gt, det):
    """TP matrix (n_det, 10) - same algorithm as ultralytics BaseValidator.match_predictions."""
    tp = np.zeros((len(det), len(IOU_T)), bool)
    if len(gt) == 0 or len(det) == 0:
        return tp
    iou = iou_matrix(gt, det)
    for i, t in enumerate(IOU_T):
        m = np.nonzero(iou >= t)
        if m[0].size:
            mm = np.stack([m[0], m[1], iou[m]], 1)
            if m[0].size > 1:
                mm = mm[mm[:, 2].argsort()[::-1]]
                mm = mm[np.unique(mm[:, 1], return_index=True)[1]]
                mm = mm[np.unique(mm[:, 0], return_index=True)[1]]
            tp[mm[:, 1].astype(int), i] = True
    return tp


def smooth(y, f=0.1):
    nf = round(len(y) * f * 2) // 2 + 1
    p = np.ones(nf // 2)
    return np.convolve(np.concatenate((p * y[0], y, p * y[-1])), np.ones(nf) / nf, mode='valid')


def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(np.concatenate(([1.0], precision, [0.0], [0.0])))))
    x = np.linspace(0, 1, 101)
    return trapezoid(np.interp(x, mrec, mpre), x)


def match_frames(dets, gts):
    """Per frame: (TP matrix, confidences, number of faces). Done once, re-used by the bootstrap."""
    return [(match_predictions(g, b), s, len(g)) for (b, s), g in zip(dets, gts)]


def metrics(matched):
    """evaluate() of finetune_blazeface_70_20_10.ipynb (all faces), on pre-matched frames."""
    tp = np.concatenate([m[0] for m in matched])
    conf = np.concatenate([m[1] for m in matched])
    n_gt = sum(m[2] for m in matched)
    order = np.argsort(-conf)
    tp, conf = tp[order], conf[order]
    tpc, fpc = tp.cumsum(0), (1 - tp).cumsum(0)
    recall, precision = tpc / (n_gt + 1e-16), tpc / (tpc + fpc + 1e-16)
    ap = np.array([compute_ap(recall[:, j], precision[:, j]) for j in range(10)]) if len(tp) else np.zeros(10)
    px = np.linspace(0, 1, 1000)
    p_curve = np.interp(-px, -conf, precision[:, 0], left=1) if len(tp) else np.ones(1000)
    r_curve = np.interp(-px, -conf, recall[:, 0], left=0) if len(tp) else np.zeros(1000)
    i = smooth(2 * p_curve * r_curve / (p_curve + r_curve + 1e-16)).argmax()
    p, r = p_curve[i], r_curve[i]
    tp_n = round(r * n_gt)
    fp_n = round(tp_n / (p + 1e-16) - tp_n)
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r + 1e-16), accuracy=p * r / (p + r - p * r + 1e-16),
                map50=ap[0], map=ap.mean(), best_conf=px[i], TP=tp_n, FP=fp_n, FN=n_gt - tp_n, faces=n_gt)


def at_conf(dets, gts, conf):
    """(n_frames, 3) TP / FP / FN per frame, keeping only detections >= conf, matched at IoU >= 0.5."""
    out = []
    for (b, s), g in zip(dets, gts):
        b = b[s >= conf]
        hit = int(match_predictions(g, b)[:, 0].sum())
        out.append((hit, len(b) - hit, len(g) - hit))
    return np.array(out, int).reshape(-1, 3)


def accuracy_of(tp, fp, fn):
    return tp / max(tp + fp + fn, 1)

## 2. Check the BlazeFace pipeline against MediaPipe

The same check as in `finetune_blazeface_70_20_10.ipynb`: on 25 frames, MediaPipe's own `FaceDetector` and the
pipeline above must find the same faces with the same scores. It compares two implementations of the same model with
each other and never looks at the labels.

In [ ]:
def mediapipe_detect(model_path, paths, min_conf):
    det = vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)), min_detection_confidence=min_conf))
    out = []
    for p in paths:
        rgb = read_rgb(p)
        h, w = rgb.shape[:2]
        res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        out.append(np.array([[d.bounding_box.origin_x / w, d.bounding_box.origin_y / h,
                              (d.bounding_box.origin_x + d.bounding_box.width) / w,
                              (d.bounding_box.origin_y + d.bounding_box.height) / h,
                              d.categories[0].score] for d in res.detections]).reshape(-1, 5))
    det.close()
    return out


conf = VIEWS['camera']['conf']
check_paths = IMAGE_PATHS['camera'][::max(1, N_FRAMES // 25)][:25]
for name in [n for n in AVAILABLE if AVAILABLE[n][0] == 'blazeface']:
    weights = AVAILABLE[name][1]
    ours = detect(*blazeface_outputs(weights, check_paths), min_score=conf)
    n_mp = n_ours = 0
    dscore, ious = [], []
    for m, (b, s) in zip(mediapipe_detect(weights, check_paths, conf), ours):
        n_mp, n_ours = n_mp + len(m), n_ours + len(b)
        if len(m) and len(b):
            iou = iou_matrix(np.clip(m[:, :4], 0, 1), b)
            for i, j in enumerate(iou.argmax(1)):
                dscore.append(abs(m[i, 4] - s[j])); ious.append(iou[i, j])
    miou = float(np.mean(ious)) if ious else 1.0
    print(f'{DISPLAY[name]}: MediaPipe {n_mp} vs this notebook {n_ours} faces on {len(check_paths)} frames, '
          f'mean |score diff| {np.mean(dscore) if dscore else 0:.4f}, mean IoU {miou:.3f}')
    assert abs(n_mp - n_ours) <= max(1, n_ours // 20) and miou > 0.9, 'pipeline does not match MediaPipe - stop here'

## 3. Run every model - once

Each model runs once on every frame of each view. Its detections are saved to `detections_<fingerprint>_<view>.npz`
on Drive together with the SHA-256 of its weights, and the run is appended to `scoring_log.csv`. On any later run the
saved detections are loaded instead, so nothing below can be "re-tested". If a model's weights on Drive have changed
since it was scored, its original detections are kept: a retrained model is a new model - add it to `MODELS` under a
new name.

YOLO / RT-DETR are also scored by Ultralytics' own `model.val()` on the same frames, as a cross-check of the label
files and the metric code.

In [ ]:
FORCE_RESCORE = False   # only to recover from a broken run; every run is logged either way
LOG_PATH = Path(OUT_DIR, 'scoring_log.csv')
LOG_FIELDS = ['scored_at', 'fingerprint', 'view', 'model', 'weights', 'weights_sha256', 'frames', 'device', 'forced',
              'ultralytics_map50', 'ultralytics_map']
RUNS = f'{DATA_ROOT}/runs'


def dets_path(view):
    return Path(OUT_DIR, f'detections_{FINGERPRINT}_{view}.npz')


def load_detections(view):
    if not dets_path(view).exists():
        return {}, {'fingerprint': FINGERPRINT, 'view': view, 'models': {}}
    z = np.load(dets_path(view))
    meta = json.loads(str(z['meta']))
    dets = {}
    for name in meta['models']:
        boxes, scores, frame = z[f'{name}__boxes'], z[f'{name}__scores'], z[f'{name}__frame']
        dets[name] = [(boxes[frame == i], scores[frame == i]) for i in range(N_FRAMES)]
    return dets, meta


def save_detections(view, dets, meta):
    arrays = {'meta': np.array(json.dumps(meta))}
    for name, per_frame in dets.items():
        arrays[f'{name}__boxes'] = np.concatenate([b for b, _ in per_frame]).reshape(-1, 4).astype(np.float32)
        arrays[f'{name}__scores'] = np.concatenate([s for _, s in per_frame]).astype(np.float32)
        arrays[f'{name}__frame'] = np.concatenate([np.full(len(s), i, np.int32) for i, (_, s) in enumerate(per_frame)])
    tmp = dets_path(view).with_name(dets_path(view).stem + '.partial.npz')
    np.savez_compressed(tmp, **arrays)
    tmp.replace(dets_path(view))


def run_model(family, weights, view):
    paths = IMAGE_PATHS[view]
    if family == 'blazeface':
        return detect(*blazeface_outputs(weights, paths)), {}
    model = (RTDETR if family == 'rtdetr' else YOLO)(weights)
    dets = []
    for i in range(0, len(paths), 16):   # Ultralytics validation settings: conf 0.001, NMS IoU 0.7
        for r in model.predict(paths[i:i + 16], imgsz=IMGSZ, conf=YOLO_CONF, iou=YOLO_IOU, max_det=MAX_DET,
                               device=DEVICE, verbose=False):
            dets.append((r.boxes.xyxyn.cpu().numpy(), r.boxes.conf.cpu().numpy()))
    data_yaml = Path(VIEWS[view]['dir'], 'webcam_test.yaml')
    data_yaml.write_text(f'path: {VIEWS[view]["dir"]}\ntrain: images\nval: images\nnames:\n  0: face\n')
    v = model.val(data=str(data_yaml), split='val', imgsz=IMGSZ, batch=16, conf=YOLO_CONF, iou=YOLO_IOU,
                  max_det=MAX_DET, device=DEVICE, plots=False, verbose=False, project=RUNS,
                  name=f'{Path(weights).parent.name}-{view}', exist_ok=True)
    return dets, {'ultralytics_map50': float(v.box.map50), 'ultralytics_map': float(v.box.map)}


DETS, META = {}, {}
new_rows = []
for view in VIEWS:
    dets, meta = load_detections(view)
    assert meta['fingerprint'] == FINGERPRINT
    for name, (family, weights) in AVAILABLE.items():
        previous = meta['models'].get(name)
        if previous and not FORCE_RESCORE:
            changed = previous['weights_sha256'] != WEIGHTS_SHA[name]
            print(f'{DISPLAY[name]} / {view}: scored {previous["scored_at"]} - re-using those detections'
                  + (' (WARNING: the weights on Drive changed since - add the new ones under a new name)'
                     if changed else ''))
            continue
        t0 = time.time()
        dets[name], extra = run_model(family, weights, view)
        meta['models'][name] = {'scored_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                                'weights': weights, 'weights_sha256': WEIGHTS_SHA[name], 'frames': N_FRAMES,
                                'device': 'GPU' if DEVICE == 0 else 'CPU', 'forced': bool(previous), **extra}
        save_detections(view, dets, meta)   # after every model, so a dropped session loses nothing
        new_rows.append({'fingerprint': FINGERPRINT, 'view': view, 'model': name, **meta['models'][name]})
        print(f'{DISPLAY[name]} / {view}: scored {N_FRAMES} frames in {time.time() - t0:.0f}s')
    DETS[view], META[view] = load_detections(view)   # always score the saved detections: identical numbers every run

if new_rows:
    log = pd.read_csv(LOG_PATH).to_dict('records') if LOG_PATH.exists() else []
    pd.DataFrame(log + new_rows, columns=LOG_FIELDS).to_csv(LOG_PATH, index=False)
assert all(DETS.values()), 'no model weights found - check YOLO_DIR, BLAZE_DIR and WEBCAM_BLAZE_DIR'
pd.read_csv(LOG_PATH)

## 4. Results

For each view, two operating points, both reported in the paper:
* **Paper protocol** - mAP plus Precision / Recall / F1 / Accuracy at each model's best-F1 confidence, exactly as in
  Table 2, so the two test sets can sit side by side.
* **At the view's fixed confidence** (camera 0.5, server 0.62) - the same threshold for every model; what the live
  system would output. Includes the share of empty-seat frames where the model reports a face anyway.

The 95% confidence intervals come from a bootstrap that resamples whole test people (frames of one person are not
independent), 1,000 times. With fewer than 5 test people it resamples frames instead, which gives narrower,
optimistic intervals - say so if you report them.

In [ ]:
CI_LEVEL = 'participant' if len(set(PARTICIPANT)) >= MIN_PARTICIPANTS_FOR_CI else 'frame'
groups = PARTICIPANT if CI_LEVEL == 'participant' else np.arange(N_FRAMES).astype(str)
if CI_LEVEL == 'frame':
    print(f'Only {len(set(PARTICIPANT))} test participant(s): confidence intervals resample frames (optimistic).')
members = [np.flatnonzero(groups == g) for g in np.unique(groups)]
rng = np.random.default_rng(SEED)
samples = [np.concatenate([members[k] for k in rng.integers(0, len(members), len(members))]) for _ in range(BOOTSTRAP)]
EMPTY = np.array([len(g) == 0 for g in GT])


def interval(values):
    lo, hi = np.percentile(values, [2.5, 97.5]) * 100
    return f'{lo:.1f}-{hi:.1f}'


rows, deploy = [], {}
for view, dets in DETS.items():
    conf = VIEWS[view]['conf']
    for name in [n for n in MODELS if n in dets]:
        matched = match_frames(dets[name], GT)
        m = metrics(matched)
        d = deploy[view, name] = at_conf(dets[name], GT, conf)
        tp, fp, fn = d.sum(0)
        boot = [(metrics([matched[i] for i in idx]), accuracy_of(*d[idx].sum(0))) for idx in samples]
        scores = np.concatenate([s for _, s in dets[name]])
        confident = scores[scores >= MEAN_CONF_MIN]
        rows.append({
            'view': view, 'model': DISPLAY[name], 'Type': TYPE[name],
            'mAP50 (%)': m['map50'] * 100, 'mAP50-95 (%)': m['map'] * 100,
            'Precision (%)': m['precision'] * 100, 'Recall (%)': m['recall'] * 100, 'F1 (%)': m['f1'] * 100,
            'Accuracy (%)': m['accuracy'] * 100, 'Accuracy 95% CI': interval([b['accuracy'] for b, _ in boot]),
            'mAP50 95% CI': interval([b['map50'] for b, _ in boot]),
            'Mean Confidence (%)': float(confident.mean() * 100) if len(confident) else 0.0,
            'TP': m['TP'], 'FP': m['FP'], 'FN': m['FN'], 'Faces': m['faces'], 'Best conf': m['best_conf'],
            'Fixed conf': conf,
            'Precision @fixed (%)': tp / max(tp + fp, 1) * 100, 'Recall @fixed (%)': tp / max(tp + fn, 1) * 100,
            'Accuracy @fixed (%)': accuracy_of(tp, fp, fn) * 100,
            'Accuracy @fixed 95% CI': interval([a for _, a in boot]),
            'TP @fixed': tp, 'FP @fixed': fp, 'FN @fixed': fn,
            'False alarms on empty frames @fixed (%)': (d[EMPTY, 1] > 0).mean() * 100 if EMPTY.any() else math.nan,
        })
        if 'ultralytics_map50' in META[view]['models'][name]:   # cross-check against Ultralytics' own validator
            u = META[view]['models'][name]['ultralytics_map50'] * 100
            flag = 'OK' if abs(u - m['map50'] * 100) < 0.5 else 'MISMATCH - check the label files before reporting'
            print(f'{DISPLAY[name]} / {view}: mAP50 {m["map50"] * 100:.2f}% here vs {u:.2f}% from model.val() - {flag}')

results = pd.DataFrame(rows).round(2)
results.to_csv(f'{OUT_DIR}/results_webcam_{FINGERPRINT}.csv', index=False)
paper_cols = ['model', 'Type', 'mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)', 'Accuracy (%)',
              'Accuracy 95% CI', 'Mean Confidence (%)', 'TP', 'FP', 'FN']
fixed_cols = ['model', 'Precision @fixed (%)', 'Recall @fixed (%)', 'Accuracy @fixed (%)', 'Accuracy @fixed 95% CI',
              'TP @fixed', 'FP @fixed', 'FN @fixed', 'False alarms on empty frames @fixed (%)']
for view in VIEWS:
    r = results[results['view'] == view]
    print(f'\nWEBCAM TEST PEOPLE - {view.upper()} VIEW ({N_FRAMES} frames, {sum(len(g) for g in GT)} faces, '
          f'{len(set(PARTICIPANT))} people; CI by {CI_LEVEL}) - paper protocol:')
    display(r[paper_cols])
    print(f'{view} view at its fixed confidence {VIEWS[view]["conf"]}:')
    display(r[fixed_cols])

In [ ]:
# Where each model fails: recall and false alarms per condition, at each view's fixed confidence.
recall_by_condition, fp_by_condition = {}, {}
counts = summary[['frames', 'faces']]
for view in VIEWS:
    cond_recall, cond_fp = {}, {}
    for (v, name), d in deploy.items():
        if v != view:
            continue
        for cond in sorted(set(CONDITION)):
            tp, fp, fn = d[CONDITION == cond].sum(0)
            cond_recall.setdefault(cond, {})[DISPLAY[name]] = tp / (tp + fn) * 100 if tp + fn else math.nan
            cond_fp.setdefault(cond, {})[DISPLAY[name]] = fp
    recall_by_condition[view] = counts.join(pd.DataFrame(cond_recall).T).round(1)
    fp_by_condition[view] = counts.join(pd.DataFrame(cond_fp).T)
    recall_by_condition[view].to_csv(f'{OUT_DIR}/recall_by_condition_{FINGERPRINT}_{view}.csv')
    fp_by_condition[view].to_csv(f'{OUT_DIR}/false_alarms_by_condition_{FINGERPRINT}_{view}.csv')
    print(f'\n{view.upper()} VIEW - recall (%) per condition at confidence {VIEWS[view]["conf"]}:')
    display(recall_by_condition[view])
    print(f'{view.upper()} VIEW - false alarms (detections that match no face) per condition:')
    display(fp_by_condition[view])

In [ ]:
# Figures - same palette and look as the paper's face-detection figures.
METRIC_COLORS = {'mAP@0.5': '#1F4E8C', 'Precision': '#2E8B57', 'F1-Score': '#D98C00', 'Recall': '#C0504D',
                 'mAP@0.5:0.95': '#6A3D9A', 'Accuracy': '#2B8C8C'}
MODEL_COLORS = {'YOLOv8n': '#1F4E8C', 'YOLOv11n': '#2E8B57', 'RT-DETR-l': '#D98C00',
                'BlazeFace (original)': '#8C8C8C', 'BlazeFace (fine-tuned)': '#C0504D',
                'BlazeFace (webcam-tuned)': '#6A3D9A'}
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'axes.labelweight': 'bold', 'axes.labelsize': 12, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})


def style(ax, xlabel=None, ylabel=None, title=None):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1.6)
    ax.grid(axis='y', color='#DDDDDD', linewidth=1)
    ax.set_axisbelow(True)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)


def legend_top(ax, ncol):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=ncol, frameon=False,
              prop={'weight': 'bold', 'size': 10})


def save(fig, fname):
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


metric_cols = [('mAP@0.5', 'mAP50 (%)'), ('mAP@0.5:0.95', 'mAP50-95 (%)'), ('Precision', 'Precision (%)'),
               ('Recall', 'Recall (%)'), ('F1-Score', 'F1 (%)'), ('Accuracy', 'Accuracy (%)')]
for view in VIEWS:
    # ---------- Test metrics across models (paper protocol), accuracy with its 95% CI ----------
    r = results[results['view'] == view].reset_index(drop=True)
    width = 0.8 / len(r)
    fig, ax = plt.subplots(figsize=(14, 6))
    for k, row in r.iterrows():
        xs = np.arange(len(metric_cols)) + (k - (len(r) - 1) / 2) * width
        vals = [row[c] for _, c in metric_cols]
        lo, hi = (float(v) for v in row['Accuracy 95% CI'].split('-'))
        err = np.zeros((2, len(metric_cols)))
        err[:, -1] = [vals[-1] - lo, hi - vals[-1]]
        ax.bar(xs, vals, width * 0.92, color=MODEL_COLORS[row['model']], edgecolor='black', linewidth=1.1,
               label=row['model'], yerr=err, capsize=3, error_kw={'linewidth': 1})
        for x, v, top in zip(xs, vals, vals[:-1] + [max(vals[-1], hi)]):   # accuracy: label above its error bar
            ax.text(x, top + 1, f'{v:.1f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold', rotation=90)
    ax.set_xticks(range(len(metric_cols)), [m for m, _ in metric_cols], fontweight='bold')
    style(ax, None, 'Score (%)')
    ax.set_ylim(0, 118)
    legend_top(ax, ncol=min(len(r), 3))
    fig.text(0.99, -0.01, f'{view.capitalize()} view. Error bars: 95% CI of Accuracy (bootstrap by {CI_LEVEL}, '
             f'{BOOTSTRAP} resamples). n = {N_FRAMES} frames, {sum(len(g) for g in GT)} faces.', ha='right',
             fontsize=8.5, color='#555555')
    save(fig, f'webcam_test_metrics_{FINGERPRINT}_{view}.png')

    # ---------- Recall per condition at the fixed confidence (frames without faces: see the false-alarm table) ----------
    with_faces = recall_by_condition[view][recall_by_condition[view]['faces'] > 0]
    conds = list(with_faces.index)
    models = [c for c in with_faces.columns if c in MODEL_COLORS]
    width = 0.8 / len(models)
    fig, ax = plt.subplots(figsize=(max(16, 1.3 * len(conds)), 6))
    for k, name in enumerate(models):
        xs = np.arange(len(conds)) + (k - (len(models) - 1) / 2) * width
        ax.bar(xs, with_faces[name], width * 0.92, color=MODEL_COLORS[name], edgecolor='black', linewidth=0.8,
               label=name)
    labels = [f'{c.replace("_", " ")}\n(n={n})' for c, n in zip(conds, with_faces['faces'])]
    ax.set_xticks(range(len(conds)), labels, fontsize=9.5)
    style(ax, None, f'Recall @ conf {VIEWS[view]["conf"]} (%) - {view} view')
    ax.set_ylim(0, 105)
    legend_top(ax, ncol=min(len(models), 3))
    save(fig, f'webcam_recall_by_condition_{FINGERPRINT}_{view}.png')
print('Figures saved to', FIG_DIR)

## 5. What to download and how to report it

Everything is in `MyDrive/knowing-eye-webcam-test/` (`<fp>` = the dataset fingerprint printed at the top, `<view>` =
`camera` or `server`):

* `results_webcam_<fp>.csv` - the table, both views: paper-protocol metrics, accuracy CI, numbers at the fixed
  confidence
* `recall_by_condition_<fp>_<view>.csv`, `false_alarms_by_condition_<fp>_<view>.csv` - where each model fails
* `figures/webcam_test_metrics_<fp>_<view>.png`, `figures/webcam_recall_by_condition_<fp>_<view>.png`
* `scoring_log.csv` - when each model was scored, on which view, with the SHA-256 of its weights (appendix)
* `detections_<fp>_<view>.npz` - every model's raw detections (what makes the results reproducible without re-testing)

**For the paper's methodology** (fill in the numbers from `label_report.md` and the tables): webcam frames of *P*
consenting participants were recorded with a laptop webcam at 640x480 under 14 conditions typical of an online exam
(frontal, reading, head turned left/right/up, sitting far/near, hand on face, phone use, movement, dim light,
backlight, a second person, an empty seat), *k* frames per condition taken on a timer. Faces were annotated by hand
following the WIDER FACE convention without access to any detector's output; unusable frames were excluded and frames
nearly identical to the previous one dropped, by fixed rules, before any model was run. Participants were assigned at
random, as a whole, to training (*a*), validation (*b*) and test (*c*) sets. All detectors were evaluated once on the
*N* test frames, both as captured and as the proctoring server receives them (480 px wide, JPEG quality 60), with the
same code and settings as the WIDER FACE / FDDB test split; 95% confidence intervals come from a participant-level
bootstrap (1,000 resamples).

Do not put participants' frames in the paper unless their consent covers publication.